# Implementation of Poisson equation in 3D (Cartesian)

In [1]:
import logging
from timeit import default_timer

import numpy as np
from matplotlib import pyplot as plt
from numba import njit
from scipy.special import erf

import multigrid3d_numba as multi

In [2]:
@njit
def DirichletDirichlet(_gridx, _gridy, _gridz, y, level):
    y[0, :, :] = 0
    y[-1, :, :] = 0
    y[:, 0, :] = 0
    y[:, -1, :] = 0
    y[:, :, 0] = 0
    y[:, :, -1] = 0

In [3]:
def setup_grid(num_xcells, num_ycells, num_zcells, x_start, x_end, y_start, y_end, z_start, z_end):

    gridx = np.linspace(x_start, x_end, num_xcells + 1)
    gridy = np.linspace(y_start, y_end, num_ycells + 1)
    gridz = np.linspace(z_start, z_end, num_zcells + 1)
    

    gridy, gridx, gridz = np.meshgrid(gridy, gridx, gridz)

    src = np.sin(gridy) * np.cos(gridx) * np.cos(gridz)
    mg = multi.Multigrid(
        num_xcells=num_xcells,
        num_ycells=num_ycells,
        num_zcells=num_zcells,
        gridx=gridx,
        gridy=gridy,
        gridz=gridz,
        src=src,
        omega_sor=1.8,
        omega_high_error_limit=1,
        omega_low_error_limit=2e-8,
        boundary=DirichletDirichlet,
        min_cells=20,
        tol=2.0e-9,
        red_black_decouple=True,
        direct_restriction=False,
    )

    mg.logger.handlers[0].setLevel(logging.DEBUG)
    mg.logger.setLevel(logging.DEBUG)
    return mg, gridx, gridy, gridz

In [4]:
mg, gridx, gridy, gridz = setup_grid(512, 512, 512, 0.0, np.pi, 0.0, np.pi, 0.0, np.pi)
start = default_timer()
mg.solve_to_tolerance(smoothing_iters=100, max_vcycle_iters=100, subcycles=2)
end = default_timer()
print(f"solve took {end - start} seconds")
print(f"solve required {mg._iterations_required} iterations on parent grid")

err=0.9999623509195722 at it=0
Starting Vcycle
 restrict at 512
 *  restrict at 256
 *  *  restrict at 128
 *  *  *  restrict at 64
 *  *  *  *  prolongate at 32
 *  *  *  restrict at 64
 *  *  *  *  prolongate at 32
 *  *  *  prolongate at 64
 *  *  restrict at 128
 *  *  *  restrict at 64
 *  *  *  *  prolongate at 32
 *  *  *  prolongate at 64
 *  *  prolongate at 128
 *  restrict at 256
 *  *  restrict at 128
 *  *  *  restrict at 64
 *  *  *  *  prolongate at 32
 *  *  *  prolongate at 64
 *  *  restrict at 128
 *  *  *  prolongate at 64
 *  *  prolongate at 128
 *  prolongate at 256
 restrict at 512
 *  restrict at 256
 *  *  restrict at 128
 *  *  *  restrict at 64
 *  *  *  *  prolongate at 32
 *  *  *  prolongate at 64
 *  *  restrict at 128
 *  *  *  prolongate at 64
 *  *  prolongate at 128
 *  restrict at 256
 *  *  restrict at 128
 *  *  *  prolongate at 64
 *  *  prolongate at 128
 *  prolongate at 256
err=3.5443675949764497e-07 at it=1
Starting Vcycle
 restrict at 512
 *

solve took 650.821886789985 seconds
solve required 800 iterations on parent grid


In [7]:
def iterate_to_solution(mg, n_iters=100, tol=2.0e-9):
    start = default_timer()
    mg.calculate_defect()
    err = mg.defect_linf
    while err > tol:
        mg.gauss_seidel_smoothing(n_iters=n_iters, omega_override=1.8)
        mg.calculate_defect()
        err = mg.defect_linf
        end = default_timer()
        print(f"solve taking at least {end - start} seconds")
        print(err)
    end = default_timer()
    print(f"solve took  {end - start} seconds")
    print(f"solve required {mg._iterations_required} iterations on parent grid")

In [ ]:
mg, gridx, gridy, gridz = setup_grid(512, 512, 512, 0.0, np.pi, 0.0, np.pi, 0.0, np.pi)

iterate_to_solution(mg, n_iters=100, tol=1.0e-8)

solve taking at least 64.155727261008 seconds
8.25746264722792
solve taking at least 127.91722460799792 seconds
7.054602437254093
solve taking at least 190.61022879800294 seconds
6.130853304333115
solve taking at least 250.5773486399994 seconds
5.379437484136846
solve taking at least 309.4124493420095 seconds
4.75059931159696
solve taking at least 368.3364546840021 seconds
4.214729170940457
solve taking at least 429.37273770700267 seconds
3.752794828299181
solve taking at least 497.1359180559957 seconds
3.350628291164088
solve taking at least 555.4149689350015 seconds
2.99837741631383
solve taking at least 613.8524084370001 seconds
2.6878588260261305
solve taking at least 674.4088972070022 seconds
2.4130729336392815
solve taking at least 734.7041089450067 seconds
2.1688935937514042
solve taking at least 794.9966235860047 seconds
1.9514139266172421
solve taking at least 855.1122881589981 seconds
1.7570046528818821
solve taking at least 914.9768754340039 seconds
1.5830200266751162
solve 